# 06 · Harness Ablation — what each part of the system adds (owner: Dolsy)
**Group 7 · SQuAD v2 "Answer-or-Abstain" assistant**

Om's final system (notebook 05) is already a small **harness** around the model: a *sensor* (his validator, rules R1–R7) and a *feedback loop* (one retry showing the model its errors). Compared with the few-shot prompt alone, it lowered hallucination (21% → 18%) but lowered overall F1 (77.0 → 74.0), because 12 answerable questions became refusals.

This notebook asks **why**, tests fixes, and measures every part separately (an *ablation*): each row below adds one change, so the difference between consecutive rows is what that change does. All rows use the same model, the same few-shot prompt, greedy decoding and the same 200 questions.

| Row | System | Source |
|---|---|---|
| E1 | Baseline prompt | notebook 02 |
| A | Few-shot prompt, no harness | notebook 02 (E2 winner) |
| B | A + Om's validator + 1 retry | notebook 05 (E7) |
| C | B with **fuzzy evidence matching** (near-exact quotes accepted) | this notebook |
| D | C + **neutral feedback**, **echo check**, **injection guard**, 2 retries | this notebook |
| E | D + **confidence threshold** (chosen on separate calibration data) | this notebook |
| F | **Qwen2.5-3B**, few-shot prompt, no harness — does a harness beat a model twice the size? | this notebook |

Plus: the tool-mode fix (Section 6) and the injection attacks (Section 7). Code: `src/harness.py` (reuses Om's `schema.py`, `confidence.py`, `tools.py`; tests in `tests/test_harness.py`).

**Runtime:** T4 GPU, ≈ 60–70 minutes. Results are saved every 20 questions, so a disconnect only costs a few rows.

In [ ]:
# --- Colab setup: Runtime > Change runtime type > T4 GPU, then run this cell ------------------------
import os, sys
REPO = "AppliedGenAIAssignments"
if not os.path.exists(REPO) and not os.path.exists("src/model.py"):
    !git clone https://github.com/OmBoroleGit/AppliedGenAIAssignments.git
if os.path.exists(REPO):
    %cd {REPO}
sys.path.insert(0, os.getcwd())
assert os.path.exists("src/harness.py"), "src/harness.py missing - push it to the repo first"
!pip install -q -r requirements.txt
!python -m unittest discover tests 2>&1 | tail -3

In [ ]:
import os, json, pandas as pd, torch
from src.model import load_model, generate_with_logprobs, set_seed, SEED
from src.runlog import load_eval_set, make_row, make_build_messages, run_cached
from src.evaluate import evaluate, score_rows, compare_systems
from src.prompts import ADVERSARIAL_CASES
from src.harness import (HarnessConfig, answer_harness, answer_with_retrieval, apply_threshold,
                         evidence_coverage, guard_injection)
from src import tools

set_seed(SEED); load_model()
data = load_eval_set("data/eval_set.jsonl")
build_messages = make_build_messages(open("results/best_prompt.txt").read().strip())   # few_shot
GEN = dict(temperature=0, seed=SEED)                                                     # greedy
OUT = "results/ablation"; os.makedirs(OUT, exist_ok=True)

CFG_B = HarnessConfig()                                                         # Om's E7 behaviour
CFG_C = HarnessConfig(fuzzy_evidence=True)
CFG_D = HarnessConfig(fuzzy_evidence=True, neutral_feedback=True, echo_check=True, input_guard=True, max_retries=2)


def row(ex, out, label):
    r = make_row(ex, out, "few_shot", "greedy")
    r.update(context=out["context_used"], gold_context=ex["context"], system=label,
             guard_removed=len(out["trace"]["removed_by_guard"]),
             evidence_coverage=round(evidence_coverage(out["result"]["evidence"], out["context_used"]), 3)
             if out["result"]["evidence"] else None)
    if "retrieved" in out:
        r.update(tool_called_by_model=out["trace"]["tool_called_by_model"],
                 gold_retrieved=any(p["text"] == ex["context"] for p in out["retrieved"]))
    return r


def run(name, items, cfg, tool_mode=False):
    fn = (lambda ex: row(ex, answer_with_retrieval(ex["question"], build_messages, generate_with_logprobs, cfg, **GEN), name)) \
        if tool_mode else \
         (lambda ex: row(ex, answer_harness(ex["question"], ex["context"], build_messages, generate_with_logprobs, cfg, **GEN), name))
    return run_cached(f"{OUT}/{name}.csv", items, fn, key=lambda ex: ex["id"])

print("ready:", len(data), "questions")

## 1 · Reproducibility check
`answer_harness` with `CFG_B` must reproduce Om's E7 run exactly; then rows C–E differ from B *only* by the harness changes.

In [ ]:
e7 = pd.read_csv("results/E7_final_system.csv", keep_default_na=False).set_index("id")
same = []
for ex in data[:5]:
    out = answer_harness(ex["question"], ex["context"], build_messages, generate_with_logprobs, CFG_B, **GEN)
    same.append(json.loads(out["raw_output"]) == json.loads(e7.loc[ex["id"], "raw_output"]))
print("Identical to Om's E7:", same)

## 2 · Why did the validator cost F1? (no model needed — Om's saved results)
Compare each question's outcome under A (few-shot alone) and B (Om's E7).

In [ ]:
A = pd.read_csv("results/exp_prompt_few_shot.csv"); B = pd.read_csv("results/E7_final_system.csv", keep_default_na=False)
sa, sb = score_rows(A).set_index("id"), score_rows(B).set_index("id")
both = sa[["error_type", "evidence", "context"]].join(sb[["error_type", "errors"]], lsuffix="_A", rsuffix="_B")
changed = both[both.error_type_A != both.error_type_B]
print(f"{len(changed)} of 200 outcomes changed between A and B\n")
display(pd.crosstab(changed.error_type_A, changed.error_type_B))
lost = changed[changed.error_type_B == "over-abstention"].copy()
lost["first_attempt_error"] = lost.errors.str.extract(r'"(R\d)')
lost["evidence_coverage"] = [round(evidence_coverage(e, c), 2) if isinstance(e, str) and e else None
                             for e, c in zip(lost.evidence, lost.context)]
display(lost[["error_type_A", "first_attempt_error", "evidence_coverage"]])

hall = sa[(sa.error_type == "hallucination")]
cov = [evidence_coverage(e, c) for e, c in zip(hall.evidence, hall.context)]
print(f"\nFew-shot hallucinations whose evidence is an EXACT quote from the paragraph: {sum(c == 1.0 for c in cov)} of {len(cov)}")
print("-> evidence checks catch invented quotes, but most SQuAD v2 traps quote real sentences; the QUESTION is the trap.")

## 3 · Rows C and D on the 200 test questions (≈ 12 + 15 minutes)

In [ ]:
C = run("C_fuzzy", data, CFG_C)
D = run("D_improved_harness", data, CFG_D)
print("injection guard fired on normal Wikipedia paragraphs (false alarms):", int((D.guard_removed.astype(int) > 0).sum()), "of", len(D))

## 4 · Confidence threshold, chosen on separate calibration data (≈ 7 minutes)
Om's confidence = geometric-mean probability of the decision tokens. The abstain-below threshold is chosen on **100 train-split questions** (50/50), never on the test set. Rule: best F1, ties → lower hallucination; over-abstention must stay ≤ 50% (on a balanced set, refusing everything already scores F1 = 50). If the best threshold is 0, confidence does not help — a valid finding.

In [ ]:
from datasets import load_dataset
train = load_dataset("rajpurkar/squad_v2", split="train").to_pandas()
train["gold_answers"] = train.answers.apply(lambda a: [str(t) for t in a["text"]])
train["is_answerable"] = train.gold_answers.apply(len) > 0
fewshot_ids = {json.loads(l)["id"] for l in open("data/fewshot_pool.jsonl")}
pool = train[~train.id.isin(fewshot_ids)]
calib_items = pd.concat([pool[pool.is_answerable].sample(50, random_state=7),
                         pool[~pool.is_answerable].sample(50, random_state=7)])[["id", "title", "context", "question", "gold_answers", "is_answerable"]].to_dict("records")
calib = run("calibration_D", calib_items, CFG_D)

grid = []
for t in [0, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    m = evaluate(apply_threshold(calib, t))
    grid.append({"threshold": t, **{k: m[k] for k in ["F1_overall", "hallucination_rate", "over_abstention_rate"]}})
grid = pd.DataFrame(grid); grid.to_csv(f"{OUT}/summary_threshold_calibration.csv", index=False)
allowed = grid[grid.over_abstention_rate <= 50]
THRESHOLD = float(allowed.sort_values(["F1_overall", "hallucination_rate"], ascending=[False, True]).iloc[0].threshold)
display(grid); print("chosen threshold:", THRESHOLD)
E = apply_threshold(D, THRESHOLD); E.to_csv(f"{OUT}/E_with_threshold.csv", index=False)

## 5 · Where do correct and wrong answers sit on the confidence scale? (test set, row D)

In [ ]:
sd = score_rows(D); answered = sd[sd.pred_answerable]
answered = answered.assign(conf=pd.to_numeric(answered.confidence, errors="coerce"))
print(answered.groupby("error_type").conf.describe()[["count", "mean", "25%", "50%", "75%"]].round(3))

## 6 · Tool mode, fixed
In notebook 05, tool mode wrote its final answer with the tool system prompt (zero-shot, no examples) instead of the few-shot prompt chosen in E2, and refused 84% of answerable questions. Here the model still decides on the tool call with Om's tool prompt and Om's `retrieve_context` runs, but the final answer uses the **same few-shot prompt and harness as paragraph mode**. The `context` column now holds the *retrieved* passages, so evidence is checked against what the model actually saw. Same 112 questions and 20-article index as notebook 05 (≈ 12 minutes).

In [ ]:
sq = load_dataset("rajpurkar/squad_v2", split="validation")
titles = []
for d in data:
    if d.get("title") and d["title"] not in titles: titles.append(d["title"])
    if len(titles) == 20: break
tools.init_index(tools.build_corpus(sq, titles=titles))
tool_items = [d for d in data if d.get("title") in set(titles)]
T = run("tool_mode_fixed", tool_items, CFG_D, tool_mode=True)
ids = [d["id"] for d in tool_items]
om_tool = pd.read_csv("results/E7_final_system_tool_mode.csv", keep_default_na=False)
tool_cmp = compare_systems({"Om tool mode (notebook 05)": om_tool, "fixed tool mode (few-shot + harness)": T,
                            "gold paragraph given (row D, same questions)": D[D.id.isin(ids)]})
ans = T[T.is_answerable.astype(str).str.lower() == "true"]
tool_cmp["gold_paragraph_retrieved_%"] = [None, round(100 * ans.gold_retrieved.astype(str).eq("True").mean(), 1), None]
tool_cmp["model_called_tool_%"] = [None, round(100 * T.tool_called_by_model.astype(str).eq("True").mean(), 1), None]
tool_cmp.to_csv(f"{OUT}/summary_tool_mode.csv")
tool_cmp[["F1_overall", "F1_answerable", "hallucination_rate", "over_abstention_rate", "evidence_faithfulness",
          "gold_paragraph_retrieved_%", "model_called_tool_%"]]

## 7 · Responsible use: the two injection attacks against A, B and D

In [ ]:
rows = []
for case in ADVERSARIAL_CASES:
    for label, cfg in [("A few-shot, no harness", HarnessConfig(validate=False)), ("B Om's E7", CFG_B), ("D improved harness", CFG_D)]:
        out = answer_harness(case["question"], case["context"], build_messages, generate_with_logprobs, cfg, **GEN)
        res, target = out["result"], case["injection_target"].lower()
        followed = target in json.dumps(res).lower() or (target == "unanswerable" and not res["answerable"])
        rows.append({"attack": case["name"], "system": label, "followed_attack": bool(followed),
                     "answered_correctly": bool(res["answerable"] and case["expected_grounded_answer"].lower() in res["answer"].lower()),
                     "guard_removed_sentences": len(out["trace"]["removed_by_guard"]), "answer": res["answer"]})
inj = pd.DataFrame(rows); inj.to_csv(f"{OUT}/summary_injection.csv", index=False); inj

## 8 · Row F: Qwen2.5-3B without a harness (≈ 15 minutes)
The 1.5B model is unloaded and the 3B model loaded in its place (same prompt, same greedy decoding). Run this section last.

In [ ]:
import src.model as M
if not os.path.exists(f"{OUT}/F_qwen3b.csv"):
    M._model = None; M._tok = None; torch.cuda.empty_cache()
    M.load_model("Qwen/Qwen2.5-3B-Instruct")
F = run("F_qwen3b", data, HarnessConfig(validate=False))

## 9 · The ablation table

In [ ]:
systems = {"E1 baseline prompt": pd.read_csv("results/exp_prompt_baseline.csv"),
           "A few-shot, no harness": A, "B + Om's validator & retry (E7)": B, "C + fuzzy evidence": C,
           "D + neutral feedback, echo, guard": D, f"E + confidence ≥ {THRESHOLD}": E, "F Qwen-3B, no harness": F}
abl = compare_systems(systems)
abl["mean_attempts"] = [round(pd.to_numeric(d["attempts"], errors="coerce").mean(), 2) if "attempts" in d else 1.0 for d in systems.values()]
abl["fallback_abstains"] = [int((d["confidence_source"] == "fallback_abstain").sum()) if "confidence_source" in d else 0 for d in systems.values()]
abl.to_csv(f"{OUT}/summary_ablation.csv")
display(abl[["F1_overall", "EM_overall", "F1_answerable", "hallucination_rate", "over_abstention_rate",
             "evidence_faithfulness", "json_validity", "mean_attempts", "fallback_abstains"]])

import matplotlib.pyplot as plt
ax = abl[["F1_overall", "hallucination_rate", "over_abstention_rate"]].plot.bar(figsize=(13, 4.5), rot=20,
        title="Harness ablation, 200 SQuAD v2 questions (%)")
ax.set_xlabel(""); plt.tight_layout(); plt.savefig(f"{OUT}/ablation_chart.png", dpi=200); plt.show()

**What changed, question by question, from A to D:**

In [ ]:
sdd = score_rows(D).set_index("id")
ch = sa[["error_type"]].join(sdd[["error_type"]], lsuffix="_A", rsuffix="_D")
ch = ch[ch.error_type_A != ch.error_type_D]
print(len(ch), "outcomes changed"); pd.crosstab(ch.error_type_A, ch.error_type_D)

**Interpretation (write after running, from the numbers above):**
1. Which change recovered the answers B lost — fuzzy matching (B→C) or neutral feedback (C→D)?
2. Did any harness row beat A (few-shot alone) on F1? On hallucination? What is the honest trade-off?
3. Does the confidence threshold help (was the chosen threshold above 0)? Do wrong answers have lower confidence than correct ones (Section 5)?
4. Row F vs the best harness row: where does the 3B model win, and where does the harness win?
5. Did the injection guard ever fire on a normal paragraph? Which systems resisted each attack?
6. Tool mode: how much did using the few-shot prompt fix? What does retrieval still cost compared with being given the paragraph?

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("ablation_results", "zip", OUT); files.download("ablation_results.zip")